# 2.1 Determining age, gender, and a person's name from a video

This notebook uses MiVOLO v2 for age and gender estimation, a face/person detector, and OpenCV YuNet/SFace models for local face matching. From this directory, install the dependencies before running the notebook:

```bash
python -m pip install setuptools==80.10.2
python -m pip install --no-build-isolation -r requirements.txt
```

The first run downloads model files from Hugging Face and the YuNet/SFace models from OpenCV Zoo into your user cache. Before running, put one clear JPG, JPEG, or PNG reference photo per person in the `photo/` directory. The filename without its extension is displayed as the name (for example, `Max Anderson.jpg` displays as `Max Anderson`). Reference images must be at least 200x200 pixels and show one clearly visible face. Faces below the similarity threshold are labeled `Unknown`.

Face matching is biometric processing. Use reference photos only with appropriate consent, protect them, and do not use the result as the sole basis for consequential decisions. The default SFace cosine-similarity threshold is 0.363. This is a score, not a probability, and may need validation for your camera and lighting conditions.

In [7]:
import hashlib
from pathlib import Path
from urllib.request import urlopen

import cv2
import torch
from transformers import AutoConfig, AutoImageProcessor, AutoModel, AutoModelForImageClassification

mivolo_revision = "53393526c220e34cdd7b722b36d22b6f9e5f4241"
detector_revision = "b3d071aedccd46a3b2d4b40609da6880a815f395"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_id = "iitolstykh/mivolo_v2"
model_config = AutoConfig.from_pretrained(
    model_id, trust_remote_code=True, revision=mivolo_revision
)
image_processor = AutoImageProcessor.from_pretrained(
    model_id, trust_remote_code=True, revision=mivolo_revision
)
age_gender_model = AutoModelForImageClassification.from_pretrained(
    model_id,
    trust_remote_code=True,
    revision=mivolo_revision,
    dtype=torch.float32,
).to(device).eval()
face_person_detector = AutoModel.from_pretrained(
    "iitolstykh/YOLO-Face-Person-Detector",
    trust_remote_code=True,
    revision=detector_revision,
    dtype=torch.float32,
)

model_cache_dir = Path.home() / ".cache" / "application_nn_processing_multimedia" / "face_models"
model_cache_dir.mkdir(parents=True, exist_ok=True)

def get_model_path(filename, url, expected_sha256):
    model_path = model_cache_dir / filename
    if model_path.is_file():
        actual_sha256 = hashlib.sha256(model_path.read_bytes()).hexdigest()
        if actual_sha256 != expected_sha256:
            raise RuntimeError(f"Cached model checksum mismatch: {model_path}")
        return model_path

    with urlopen(url, timeout=60) as response:
        model_bytes = response.read()
    actual_sha256 = hashlib.sha256(model_bytes).hexdigest()
    if actual_sha256 != expected_sha256:
        raise RuntimeError(f"Downloaded model checksum mismatch: {filename}")
    model_path.write_bytes(model_bytes)
    return model_path

yunet_model_path = get_model_path(
    "face_detection_yunet_2023mar.onnx",
    "https://github.com/opencv/opencv_zoo/raw/refs/heads/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx",
    "8f2383e4dd3cfbb4553ea8718107fc0423210dc964f9f4280604804ed2552fa4",
)
sface_model_path = get_model_path(
    "face_recognition_sface_2021dec.onnx",
    "https://github.com/opencv/opencv_zoo/raw/refs/heads/main/models/face_recognition_sface/face_recognition_sface_2021dec.onnx",
    "0ba9fbfa01b5270c96627c4ef784da859931e02f04419c829e83484087c34e79",
)
face_detector = cv2.FaceDetectorYN_create(
    str(yunet_model_path), "", (320, 320), 0.5, 0.3, 5000
)
face_recognizer = cv2.FaceRecognizerSF_create(str(sface_model_path), "")

def extract_face_feature(image):
    height, width = image.shape[:2]
    if width < 20 or height < 20:
        return None
    face_detector.setInputSize((width, height))
    _, detected_faces = face_detector.detect(image)
    if detected_faces is None or len(detected_faces) == 0:
        return None
    face = max(detected_faces, key=lambda row: row[2] * row[3])
    aligned_face = face_recognizer.alignCrop(image, face)
    return face_recognizer.feature(aligned_face)

photo_dir = Path("photo")
if not photo_dir.is_dir() and (Path.cwd() / "02_determining_gender_age_video" / "photo").is_dir():
    photo_dir = Path.cwd() / "02_determining_gender_age_video" / "photo"
if not photo_dir.is_dir():
    raise FileNotFoundError(f"Reference-photo directory not found: {photo_dir.resolve()}")

supported_extensions = {".jpg", ".jpeg", ".png"}
reference_files = sorted(
    path
    for path in photo_dir.iterdir()
    if path.is_file() and path.suffix.lower() in supported_extensions
)
if not reference_files:
    raise RuntimeError(f"No JPG, JPEG, or PNG reference photos found in {photo_dir.resolve()}")

reference_features = []
for path in reference_files:
    image = cv2.imread(str(path))
    if image is None:
        print(f"Skipping unreadable reference photo: {path.name}")
        continue
    if min(image.shape[:2]) < 200:
        print(f"Skipping reference photo smaller than 200x200 pixels: {path.name}")
        continue

    feature = extract_face_feature(image)
    if feature is None:
        print(f"Skipping reference photo with no detectable face: {path.name}")
        continue
    reference_features.append((path.stem, feature))

if not reference_features:
    raise RuntimeError(
        f"No usable reference faces found in {photo_dir.resolve()}. "
        "Add readable photos (at least 200x200 pixels) with a clearly visible face."
    )
print(f"Loaded {len(reference_features)} reference face(s) from {photo_dir.resolve()}")

recognition_threshold = 0.363

def identify_face(face_crop):
    feature = extract_face_feature(face_crop)
    if feature is None:
        return "Unknown"

    best_name, best_similarity = max(
        (
            (
                name,
                float(
                    face_recognizer.match(
                        feature, reference_feature, cv2.FaceRecognizerSF_FR_COSINE
                    )
                ),
            )
            for name, reference_feature in reference_features
        ),
        key=lambda match: match[1],
    )
    if best_similarity < recognition_threshold:
        return "Unknown"
    return best_name

def analyze_frame(frame):
    detections = face_person_detector(
        frame, conf=0.4, iou=0.7, verbose=False, device=str(device)
    )[0]
    face_boxes = []
    person_boxes = []
    frame_height, frame_width = frame.shape[:2]

    for detection in detections.boxes:
        label = detections.names[int(detection.cls.item())]
        x1, y1, x2, y2 = [int(round(value)) for value in detection.xyxy[0].tolist()]
        box = (
            max(0, min(x1, frame_width)),
            max(0, min(y1, frame_height)),
            max(0, min(x2, frame_width)),
            max(0, min(y2, frame_height)),
        )
        if box[2] <= box[0] or box[3] <= box[1]:
            continue
        if label == "face":
            face_boxes.append(box)
        elif label == "person":
            person_boxes.append(box)

    face_crops = []
    person_crops = []
    matched_face_boxes = []
    for face_box in face_boxes:
        face_x1, face_y1, face_x2, face_y2 = face_box
        center_x = (face_x1 + face_x2) / 2
        center_y = (face_y1 + face_y2) / 2
        matching_people = [
            box
            for box in person_boxes
            if box[0] <= center_x <= box[2] and box[1] <= center_y <= box[3]
        ]
        if not matching_people:
            continue
        person_box = min(
            matching_people,
            key=lambda box: (box[2] - box[0]) * (box[3] - box[1]),
        )
        person_x1, person_y1, person_x2, person_y2 = person_box
        face_crops.append(frame[face_y1:face_y2, face_x1:face_x2])
        person_crops.append(frame[person_y1:person_y2, person_x1:person_x2])
        matched_face_boxes.append(face_box)

    if not face_crops:
        return []

    faces_input = image_processor(images=face_crops)[
        "pixel_values"
    ].to(device=device, dtype=age_gender_model.dtype)
    body_input = image_processor(images=person_crops)[
        "pixel_values"
    ].to(device=device, dtype=age_gender_model.dtype)
    with torch.inference_mode():
        output = age_gender_model(faces_input=faces_input, body_input=body_input)

    predictions = []
    for index, face_box in enumerate(matched_face_boxes):
        gender_index = int(output.gender_class_idx[index].item())
        gender = model_config.gender_id2label[gender_index]
        age = float(output.age_output[index].item())
        name = identify_face(face_crops[index])
        predictions.append((face_box, age, gender, name))
    return predictions


camera = cv2.VideoCapture(0)
if not camera.isOpened():
    camera.release()
    raise RuntimeError("Cannot open the webcam. Check the connection and permissions.")

camera.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
camera.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
actual_width = int(camera.get(cv2.CAP_PROP_FRAME_WIDTH))
actual_height = int(camera.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Camera resolution: {actual_width}x{actual_height}")

analysis_interval = 15
frame_number = 0
predictions = []
window_name = "Age, gender and face recognition"
print("Age and gender are estimated from appearance; face names are approximate matches. Press q to exit.")

try:
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    frame_width = int(camera.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(camera.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cv2.resizeWindow(window_name, frame_width, frame_height)
    while True:
        success, frame = camera.read()
        if not success:
            raise RuntimeError("Failed to capture a frame from the webcam.")

        frame_number += 1
        if frame_number == 1 or frame_number % analysis_interval == 0:
            predictions = analyze_frame(frame)

        if not predictions:
            cv2.putText(
                frame,
                "Face/person not detected",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 0, 255),
                2,
            )

        for face_box, age, gender, name in predictions:
            x1, y1, x2, y2 = face_box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            label = f"{name}, ~{age:.0f}, {gender}"
            cv2.putText(
                frame,
                label,
                (x1, max(y1 - 10, 25)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2,
            )

        cv2.imshow(window_name, frame)
        key = cv2.waitKey(1) & 0xFF
        try:
            window_open = cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) >= 1
        except cv2.error:
            window_open = False
        if key == ord("q") or not window_open:
            break
finally:
    camera.release()
    cv2.destroyAllWindows()

Overriding model.yaml nc=80 with nc=2
YOLOv8x summary (fused): 268 layers, 68125494 parameters, 0 gradients, 257.4 GFLOPs


[ WARN:0@6592.303] global net_impl_backend.cpp:345 setPreferableTarget Targets are not supported by the new graph engine for now
[ WARN:0@6592.393] global net_impl_backend.cpp:345 setPreferableTarget Targets are not supported by the new graph engine for now


Loaded 1 reference face(s) from /home/max/Projects/application_nn_processing_multimedia/02_determining_gender_age_video/photo
Camera resolution: 1920x1080
Age and gender are estimated from appearance; face names are approximate matches. Press q to exit.
